# Snowflake Iceberg Interop Explorer — Complete Story (25 Features)

**Import into Snowsight:** Projects → Notebooks → Import → select this file

Uses `get_active_session()` — zero auth setup, works out of the box.

| # | Feature | Category |
|---|---------|----------|
| 1 | Open Iceberg REST Access | Core |
| 2 | Apache Polaris Integration | Core |
| 3 | Single Endpoint | Core |
| 4 | External Engine Read + Write | Interop |
| 5 | Snowflake Security Model | Governance |
| 6 | Credential Vending | Governance |
| 7 | Governed Multi-Engine Access | Governance |
| 8 | Policy Enforcement on Iceberg | Governance |
| 9 | Supported External Engines | Reference |
| 10 | Snowflake Storage for Iceberg | Storage |
| 11 | AWS Glue + Catalog-Linked Databases | Catalog |
| 12 | Iceberg Time Travel | Data Management |
| 13 | Table Maintenance (OPTIMIZE/REORG) | Operations |
| 14 | Schema Evolution | Operations |
| 15 | Competitive Positioning | Reference |
| 16 | Snowpipe Streaming → Iceberg | Ingestion |
| 17 | Dynamic Tables as Iceberg | Pipelines |
| 18 | Partitioning + Performance Tuning | Performance |
| 19 | Secure Data Sharing for Iceberg | Sharing |
| 20 | Delta Sharing Protocol | Sharing |
| 21 | Snowpark on Iceberg | Native Python |
| 22 | Object Tags + Data Classification | Governance |
| 23 | Unity Catalog ↔ Horizon Catalog | Multi-Cloud |
| 24 | PrivateLink for Catalog Integrations | Security |
| 25 | Iceberg v3 Features | Format |

## 0. Setup — Configure your account details, create demo database + Iceberg table

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd, json

session = get_active_session()

# ── CONFIGURE THESE ───────────────────────────────────────────────────────────
EXTERNAL_VOLUME = 'iceberg_demo_ext_vol'  # your external volume (for Features 1-9, 11-18)
DATABASE        = 'horizon_demo_db'        # will be created
SCHEMA          = 'public'
WAREHOUSE       = 'COMPUTE_WH'
# ─────────────────────────────────────────────────────────────────────────────

session.sql(f'CREATE DATABASE IF NOT EXISTS {DATABASE}').collect()
session.sql(f'CREATE SCHEMA IF NOT EXISTS {DATABASE}.{SCHEMA}').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'USE SCHEMA {SCHEMA}').collect()

HORIZON_URI = session.sql("SELECT SYSTEM$GET_ICEBERG_REST_CATALOG_ENDPOINT() AS e").to_pandas()['E'].iloc[0]
print(f"Account  : {session.get_current_account()}")
print(f"Horizon  : {HORIZON_URI}")
print(f"Database : {DATABASE}.{SCHEMA}")

In [ ]:
session.sql(f'''
CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions (
    transaction_id   VARCHAR(36),  customer_id VARCHAR(36),
    amount           DECIMAL(12,2), currency   VARCHAR(3),
    transaction_ts   TIMESTAMP_NTZ(6),
    status           VARCHAR(20),  region      VARCHAR(50)
)
    CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}'
    BASE_LOCATION='horizon_demo/transactions/'
''').collect()

session.sql(f"""
INSERT INTO {DATABASE}.{SCHEMA}.transactions VALUES
    ('txn-001','cust-A',1250.00,'USD','2024-01-15 09:30:00'::TIMESTAMP_NTZ(6),'COMPLETED','us-west'),
    ('txn-002','cust-B', 320.50,'USD','2024-01-15 10:15:00'::TIMESTAMP_NTZ(6),'PENDING',  'us-east'),
    ('txn-003','cust-A', 875.00,'EUR','2024-01-15 11:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','eu-west'),
    ('txn-004','cust-C',4200.00,'USD','2024-01-15 12:45:00'::TIMESTAMP_NTZ(6),'FAILED',   'us-west'),
    ('txn-005','cust-B', 650.75,'GBP','2024-01-15 13:30:00'::TIMESTAMP_NTZ(6),'COMPLETED','eu-west'),
    ('txn-006','cust-D', 980.00,'USD','2024-01-16 08:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','us-east'),
    ('txn-007','cust-E', 450.00,'EUR','2024-01-16 09:15:00'::TIMESTAMP_NTZ(6),'PENDING',  'eu-west'),
    ('txn-008','cust-F',1100.00,'USD','2024-01-17 10:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','us-west'),
    ('txn-009','cust-G', 220.50,'GBP','2024-01-17 11:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','eu-west'),
    ('txn-010','cust-A', 340.00,'USD','2024-01-18 08:30:00'::TIMESTAMP_NTZ(6),'FAILED',   'us-east')
""").collect()

session.sql(f"SELECT * FROM {DATABASE}.{SCHEMA}.transactions").to_pandas()

---
## Feature 1 — Open Iceberg REST Access
Horizon Catalog exposes every Iceberg table via the **Apache Iceberg REST Catalog API**. Any engine that speaks the open REST protocol can query Snowflake tables — no proprietary connector needed.

In [ ]:
print(f"Horizon Catalog REST endpoint:\n  {HORIZON_URI}\n")
print("External engines connect using:")
print(f"  PyIceberg : RestCatalog(uri='{HORIZON_URI}', token=<session_token>)")
print(f"  Spark     : spark.sql.catalog.sf.uri = {HORIZON_URI}")
print(f"  DuckDB    : ATTACH '{HORIZON_URI}' AS h (TYPE ICEBERG_REST, ...)")
print(f"  Trino     : iceberg.rest-catalog.uri={HORIZON_URI}")

# Iceberg metadata for the table
info_raw = session.sql(f"SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('{DATABASE}.{SCHEMA}.transactions') AS i").to_pandas()['I'].iloc[0]
info = json.loads(info_raw)
print("\nIceberg table metadata:")
for k,v in info.items():
    print(f"  {k}: {str(v)[:80]}")

---
## Feature 2 — Apache Polaris Integration
Snowflake Horizon implements the same open REST protocol as Apache Polaris.
It can **serve as** a Polaris endpoint OR **read tables from** an external Polaris instance.

In [ ]:
print("Direction A — Snowflake IS the Polaris-compatible endpoint:")
print(f"  External Spark connects to: {HORIZON_URI}")
print()
print("Direction B — Snowflake reads an EXTERNAL Polaris instance:")
print("""
CREATE CATALOG INTEGRATION polaris_int
    CATALOG_SOURCE = POLARIS  TABLE_FORMAT = ICEBERG
    REST_CONFIG = (
        CATALOG_URI = 'http://<polaris-host>:8181/api/catalog'
        WAREHOUSE   = 'my_catalog'
    )
    REST_AUTHENTICATION = (
        TYPE = OAUTH
        OAUTH_TOKEN_URI     = 'http://<polaris-host>:8181/api/catalog/v1/oauth/tokens'
        OAUTH_CLIENT_ID     = '<client_id>'
        OAUTH_CLIENT_SECRET = '<client_secret>'
        OAUTH_ALLOWED_SCOPES = ('PRINCIPAL_ROLE:ALL')
    ) ENABLED = TRUE;

CREATE DATABASE polaris_db
    LINKED_CATALOG = (CATALOG_INTEGRATION = 'polaris_int')
    EXTERNAL_VOLUME = 'my_ext_vol';
""")
session.sql("SHOW INTEGRATIONS LIKE '%CATALOG%'").to_pandas()

---
## Features 3–9: Single Endpoint · Read/Write · Security · Credential Vending · Governance · Policies · Engines

In [ ]:
# Feature 3 — Single Endpoint
print("=== Feature 3: Single Endpoint ===")
tables = session.sql("SELECT table_catalog, table_schema, table_name FROM information_schema.tables WHERE table_type='ICEBERG' ORDER BY 1,2,3").to_pandas()
print(f"All Iceberg tables served from one URI ({HORIZON_URI}):")
print(tables.to_string(index=False))

# Feature 4 — Read + Write
print("\n=== Feature 4: External Engine Read + Write ===")
session.sql(f"INSERT INTO {DATABASE}.{SCHEMA}.transactions VALUES ('txn-spark','cust-X',750.00,'USD','2024-01-19 09:00:00'::TIMESTAMP_NTZ(6),'COMPLETED','us-west')").collect()
session.sql(f"MERGE INTO {DATABASE}.{SCHEMA}.transactions t USING (SELECT 'txn-spark' AS id,'FAILED' AS s) src ON t.transaction_id=src.id WHEN MATCHED THEN UPDATE SET t.status=src.s").collect()
session.sql(f"DELETE FROM {DATABASE}.{SCHEMA}.transactions WHERE status='FAILED'").collect()
count = session.sql(f"SELECT COUNT(*) AS cnt FROM {DATABASE}.{SCHEMA}.transactions").to_pandas()['CNT'].iloc[0]
print(f"After INSERT, MERGE, DELETE: {count} rows remain")

# Feature 5 — Security Model
print("\n=== Feature 5: Security Model ===")
for sql in [
    "CREATE ROLE IF NOT EXISTS iceberg_ext_role",
    f"GRANT USAGE ON DATABASE {DATABASE} TO ROLE iceberg_ext_role",
    f"GRANT USAGE ON SCHEMA {DATABASE}.{SCHEMA} TO ROLE iceberg_ext_role",
    f"GRANT SELECT, INSERT ON TABLE {DATABASE}.{SCHEMA}.transactions TO ROLE iceberg_ext_role",
]:
    session.sql(sql).collect()
print("✓ Service role created with scoped Iceberg grants")
print(f"  Token scope for external engine: session:role:iceberg_ext_role @ {HORIZON_URI}")

In [ ]:
# Feature 6 — Credential Vending
print("=== Feature 6: Credential Vending ===")
vol = session.sql(f"DESC EXTERNAL VOLUME {EXTERNAL_VOLUME}").to_pandas()
iam_arn = vol[vol['property'].str.contains('IAM_USER_ARN', na=False)]['property_value'].values
print(f"External volume IAM ARN: {iam_arn[0] if len(iam_arn) else 'see DESC EXTERNAL VOLUME output'}")
print("When engine sends X-Iceberg-Access-Delegation: vended-credentials, Horizon returns:")
print("  s3.access-key-id  (temp, ~15 min TTL)")
print("  s3.secret-access-key")
print("  s3.session-token")

# Feature 7 — Governed Multi-Engine
print("\n=== Feature 7: Governed Multi-Engine Access ===")
for role in ['spark_analyst_role','trino_analyst_role','duckdb_analyst_role']:
    session.sql(f"CREATE ROLE IF NOT EXISTS {role}").collect()
    session.sql(f"GRANT USAGE ON DATABASE {DATABASE} TO ROLE {role}").collect()
    session.sql(f"GRANT USAGE ON SCHEMA {DATABASE}.{SCHEMA} TO ROLE {role}").collect()
    session.sql(f"GRANT SELECT ON TABLE {DATABASE}.{SCHEMA}.transactions TO ROLE {role}").collect()
print("✓ spark_analyst_role, trino_analyst_role, duckdb_analyst_role — all see identical metadata")
print("  Revoking any role instantly revokes access across ALL engines")

# Feature 8 — Policy Enforcement
print("\n=== Feature 8: Policy Enforcement ===")
session.sql(f"""
CREATE OR REPLACE ROW ACCESS POLICY {DATABASE}.{SCHEMA}.rap_region AS (region_col VARCHAR)
RETURNS BOOLEAN ->
    CASE WHEN CURRENT_ROLE()='ANALYST_US' THEN region_col LIKE 'us-%'
         WHEN CURRENT_ROLE()='ANALYST_EU' THEN region_col LIKE 'eu-%'
         ELSE CURRENT_ROLE() IN ('ACCOUNTADMIN','SYSADMIN') END
""").collect()
session.sql(f"ALTER TABLE {DATABASE}.{SCHEMA}.transactions ADD ROW ACCESS POLICY {DATABASE}.{SCHEMA}.rap_region ON (region)").collect()
session.sql(f"CREATE OR REPLACE MASKING POLICY {DATABASE}.{SCHEMA}.mask_customer AS (val VARCHAR) RETURNS VARCHAR -> CASE WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN','SYSADMIN') THEN val ELSE SHA2(val,256) END").collect()
session.sql(f"ALTER TABLE {DATABASE}.{SCHEMA}.transactions MODIFY COLUMN customer_id SET MASKING POLICY {DATABASE}.{SCHEMA}.mask_customer").collect()
print("✓ Row access policy (region filter by role) + masking policy (customer_id hash) applied")
print("  These policies enforce on Spark/DuckDB queries through Horizon — transparently")
session.sql(f"SELECT policy_name,policy_kind,ref_column_name FROM TABLE(information_schema.policy_references(ref_entity_name=>'{DATABASE}.{SCHEMA}.transactions',ref_entity_domain=>'TABLE'))").to_pandas()

In [ ]:
# Feature 9 — Supported Engines reference table
print("=== Feature 9: Supported External Engines ===")
engines = pd.DataFrame([
    ('Apache Spark',  'iceberg-spark-runtime-3.5_2.12:1.7.0','Read+Write','spark.sql.catalog.sf.type=rest'),
    ('Apache Flink',  'iceberg-flink-runtime-1.18:1.7.0',    'Read+Write',"'catalog-type'='rest'"),
    ('Trino',         'Built-in (>=430)',                     'Read+Write','iceberg.rest-catalog.uri=...'),
    ('DuckDB',        'iceberg extension (>=1.1.0)',          'Read',      'ATTACH ... TYPE ICEBERG_REST'),
    ('PyIceberg',     'pyiceberg[rest]',                      'Read+Write','RestCatalog(uri=..., token=...)'),
    ('Dremio',        'Native Iceberg REST',                  'Read+Write','"type":"ICEBERG_REST"'),
    ('Apache Doris',  'Built-in (>=2.1)',                     'Read+Write',"'iceberg.catalog.type'='rest'"),
    ('StarRocks',     'Built-in (>=3.2)',                     'Read+Write','iceberg.catalog.rest.uri=...'),
], columns=['Engine','Library','DML','Key Config'])
display(engines)

---
## Features 10–18: Storage · Glue · Time Travel · Maintenance · Schema Evolution · Competitive · Streaming · Dynamic Tables · Partitioning

In [ ]:
# Feature 10 — Snowflake Storage for Iceberg
print("=== Feature 10: Snowflake Storage for Iceberg ===")
print("No S3 bucket, no IAM role, no external volume — just STORAGE_SERIALIZATION_POLICY = COMPATIBLE")
session.sql(f"""
CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.orders_sf_storage (
    order_id VARCHAR(36), customer_id VARCHAR(36), order_date DATE,
    total_amount DECIMAL(12,2), status VARCHAR(20), region VARCHAR(50)
)   CATALOG='SNOWFLAKE' STORAGE_SERIALIZATION_POLICY=COMPATIBLE
""").collect()
session.sql(f"""
INSERT INTO {DATABASE}.{SCHEMA}.orders_sf_storage VALUES
    ('ord-001','cust-A','2024-01-15',1250.00,'SHIPPED','us-west'),
    ('ord-002','cust-B','2024-01-16', 320.50,'PENDING','us-east'),
    ('ord-003','cust-C','2024-01-17',4200.00,'DELIVERED','eu-west')
""").collect()
display(session.sql(f"SELECT * FROM {DATABASE}.{SCHEMA}.orders_sf_storage").to_pandas())

info = json.loads(session.sql(f"SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('{DATABASE}.{SCHEMA}.orders_sf_storage') AS i").to_pandas()['I'].iloc[0])
print(f"Still open Iceberg via Horizon: {info.get('metadataLocation','')[:60]}...")

# Feature 11 — AWS Glue + Catalog-Linked DB
print("\n=== Feature 11: AWS Glue + Catalog-Linked Databases ===")
print("""
CREATE CATALOG INTEGRATION glue_catalog_int
    CATALOG_SOURCE = ICEBERG_REST  TABLE_FORMAT = ICEBERG
    REST_CONFIG = (
        CATALOG_URI      = 'https://glue.us-east-2.amazonaws.com/iceberg'
        CATALOG_API_TYPE = AWS_GLUE   WAREHOUSE = '123456789012'
        CATALOG_NAME     = '123456789012'
        ACCESS_DELEGATION_MODE = EXTERNAL_VOLUME_CREDENTIALS
    ) ENABLED = TRUE;

CREATE DATABASE glue_linked_db
    LINKED_CATALOG = (CATALOG_INTEGRATION = 'glue_catalog_int')
    EXTERNAL_VOLUME = 'my_ext_vol';
-- Auto-discovers all Glue databases as schemas; query with lowercase + double quotes
SELECT * FROM glue_linked_db."my_glue_db"."my_table" LIMIT 10;
""")

In [ ]:
# Feature 12 — Iceberg Time Travel
print("=== Feature 12: Iceberg Time Travel ===")
snapshots = session.sql(f"""
SELECT snapshot_id, committed_at, operation
FROM TABLE(information_schema.iceberg_table_history(
    TABLE_NAME=>'{DATABASE}.{SCHEMA}.transactions',
    DATABASE_NAME=>'{DATABASE.upper()}', SCHEMA_NAME=>'{SCHEMA.upper()}'
)) ORDER BY committed_at DESC
""").to_pandas()
display(snapshots.head(5))

if len(snapshots) > 0:
    snap_id = snapshots['SNAPSHOT_ID'].iloc[-1]
    print(f"\nTime travel to earliest snapshot ({snap_id}):")
    df_old = session.sql(f"SELECT COUNT(*) AS cnt FROM {DATABASE}.{SCHEMA}.transactions AT(SNAPSHOT => {snap_id})").to_pandas()
    print(f"  Row count at that snapshot: {df_old['CNT'].iloc[0]}")

print("\nTime travel 5 minutes ago:")
df_past = session.sql(f"SELECT COUNT(*) AS cnt FROM {DATABASE}.{SCHEMA}.transactions AT(TIMESTAMP => DATEADD(MINUTE,-5,CURRENT_TIMESTAMP()))").to_pandas()
print(f"  Row count: {df_past['CNT'].iloc[0]}")

# Demonstrate recovery
print("\nAccidental delete + recovery with BEFORE(STATEMENT):")
session.sql(f"DELETE FROM {DATABASE}.{SCHEMA}.transactions WHERE status='PENDING'").collect()
count_after = session.sql(f"SELECT COUNT(*) AS c FROM {DATABASE}.{SCHEMA}.transactions").to_pandas()['C'].iloc[0]
session.sql(f"INSERT INTO {DATABASE}.{SCHEMA}.transactions SELECT * FROM {DATABASE}.{SCHEMA}.transactions BEFORE(STATEMENT=>LAST_QUERY_ID()) WHERE status='PENDING'").collect()
count_recovered = session.sql(f"SELECT COUNT(*) AS c FROM {DATABASE}.{SCHEMA}.transactions").to_pandas()['C'].iloc[0]
print(f"  After delete: {count_after} rows  |  After recovery: {count_recovered} rows")

In [ ]:
# Feature 13 — Table Maintenance
print("=== Feature 13: Table Maintenance ===")
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions OPTIMIZE").collect()
print("✓ OPTIMIZE — compacted small Parquet files (run after bulk Spark/Flink writes)")
session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.transactions EXPIRE SNAPSHOTS OLDER_THAN = DATEADD(DAY,-7,CURRENT_TIMESTAMP())").collect()
print("✓ EXPIRE SNAPSHOTS — removed metadata older than 7 days")
print("  Tip: ALTER ICEBERG TABLE ... SET AUTO_REFRESH = TRUE for automatic background compaction")

# Feature 14 — Schema Evolution
print("\n=== Feature 14: Schema Evolution ===")
print("Before:", [r[0] for r in session.sql(f"DESCRIBE TABLE {DATABASE}.{SCHEMA}.transactions").collect()])
session.sql(f"ALTER TABLE {DATABASE}.{SCHEMA}.transactions ADD COLUMN merchant_id VARCHAR(36)").collect()
session.sql(f"ALTER TABLE {DATABASE}.{SCHEMA}.transactions RENAME COLUMN merchant_id TO merchant_ref_id").collect()
session.sql(f"ALTER TABLE {DATABASE}.{SCHEMA}.transactions ADD COLUMN loyalty_pts INT DEFAULT 0").collect()
session.sql(f"ALTER TABLE {DATABASE}.{SCHEMA}.transactions DROP COLUMN loyalty_pts").collect()
print("After  :", [r[0] for r in session.sql(f"DESCRIBE TABLE {DATABASE}.{SCHEMA}.transactions").collect()])
print("All column changes are metadata-only — no Parquet files rewritten")
print("External engines (Spark ALTER TABLE) can also evolve the schema; Snowflake picks it up automatically")

In [ ]:
# Feature 15 — Competitive Positioning
print("=== Feature 15: Competitive Positioning ===")
comp = pd.DataFrame([
    ('Snowflake + Horizon', 'Full DML',    'Native AT/BEFORE',   'RAP + Masking all engines', 'Yes',   'Open REST', 'GA'),
    ('Databricks + UC',     'Full DML',    'Delta only (native)', 'Unity Catalog only',        'Cond.', 'Open REST', 'GA w/ caveats'),
    ('AWS Glue',            'Full DML',    'Via IAM',             'Lake Formation',             'Yes',   'Open REST', 'GA'),
    ('Google BigLake',      'Read-heavy',  'Via IAM',             'BigQuery policies',          'Ltd.',  'Open REST', 'Evolving'),
    ('Microsoft Fabric',    'Delta focus', 'Via Entra ID',        'Fabric policies',            'Ltd.',  'Partial',   'Evolving'),
], columns=['Platform','DML','Time Travel','Policy Enforcement','Cred. Vending','Iceberg REST','Status'])
display(comp)
print()
print("Key Snowflake differentiators:")
for d in [
    "RAP + Masking enforced on Spark/DuckDB queries via Horizon — unique to Snowflake",
    "Native time travel (AT/BEFORE) on Iceberg tables, not just Delta",
    "OPTIMIZE, Dynamic Tables, Snowpipe Streaming all produce open Iceberg output",
    "Single governed endpoint — one URI, one RBAC system, all engines",
]:
    print(f"  ✓ {d}")

In [ ]:
# Feature 16 — Snowpipe Streaming → Iceberg
print("=== Feature 16: Snowpipe Streaming → Iceberg ===")
session.sql(f"""
CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.clickstream (
    event_id VARCHAR(36), user_id VARCHAR(36), event_type VARCHAR(50),
    page VARCHAR(200), event_ts TIMESTAMP_NTZ(6), region VARCHAR(50)
)   CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}' BASE_LOCATION='horizon_demo/clickstream/'
""").collect()

import uuid
from datetime import datetime, timezone
rows = [(str(uuid.uuid4()), str(uuid.uuid4()), 'click', f'/product/{i}', datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S'), 'us-west') for i in range(10)]
vals = ','.join([f"('{r[0]}','{r[1]}','{r[2]}','{r[3]}','{r[4]}'::TIMESTAMP_NTZ(6),'{r[5]}')" for r in rows])
session.sql(f"INSERT INTO {DATABASE}.{SCHEMA}.clickstream VALUES {vals}").collect()
count = session.sql(f"SELECT COUNT(*) AS c FROM {DATABASE}.{SCHEMA}.clickstream").to_pandas()['C'].iloc[0]
print(f"✓ {count} events in Iceberg clickstream table (simulates Snowpipe Streaming)")
print("\nKafka Connector config key:")
print("  snowflake.ingestion.method=SNOWPIPE_STREAMING")
print("  snowflake.output.schema.evolution=TRUE")
print("  External engines read streamed data via Horizon REST immediately after commit")

# Feature 17 — Dynamic Tables as Iceberg
print("\n=== Feature 17: Dynamic Tables as Iceberg ===")
session.sql(f"""
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_daily_agg
    TARGET_LAG=DOWNSTREAM WAREHOUSE={WAREHOUSE}
    CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}' BASE_LOCATION='horizon_demo/txn_daily_agg/'
AS
    SELECT DATE_TRUNC('day', transaction_ts) AS txn_date, region, currency,
           COUNT(*) AS txn_count, ROUND(SUM(amount),2) AS total_amount
    FROM {DATABASE}.{SCHEMA}.transactions GROUP BY 1,2,3
""").collect()
session.sql(f"ALTER DYNAMIC TABLE {DATABASE}.{SCHEMA}.txn_daily_agg REFRESH").collect()
print("✓ DYNAMIC ICEBERG TABLE created and refreshed")
display(session.sql(f"SELECT * FROM {DATABASE}.{SCHEMA}.txn_daily_agg ORDER BY txn_date").to_pandas())

In [ ]:
# Feature 18 — Partitioning + Performance Tuning
print("=== Feature 18: Partitioning + Performance Tuning ===")
session.sql(f"""
CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_partitioned (
    transaction_id VARCHAR(36), customer_id VARCHAR(36),
    amount DECIMAL(12,2), currency VARCHAR(3),
    transaction_ts TIMESTAMP_NTZ(6), status VARCHAR(20), region VARCHAR(50)
)
    CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}' BASE_LOCATION='horizon_demo/txn_partitioned/'
    PARTITION BY ( MONTH(transaction_ts), IDENTITY(region) )
""").collect()
print("✓ Table partitioned by MONTH(transaction_ts) + IDENTITY(region)")

session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_partitioned SET PARTITION SPEC (YEAR(transaction_ts), IDENTITY(region))").collect()
print("✓ Partition evolution: MONTH → YEAR (zero downtime, old files unchanged)")

session.sql(f"ALTER ICEBERG TABLE {DATABASE}.{SCHEMA}.txn_partitioned SET SORT ORDER (region ASC NULLS LAST, transaction_ts DESC NULLS LAST)").collect()
print("✓ Sort order set: new files sorted for better predicate pruning")

print()
transforms = pd.DataFrame([
    ('IDENTITY(region)',        'Exact-match pruning — low cardinality'),
    ('MONTH(transaction_ts)',   'Date-range pruning — time series'),
    ('BUCKET(16, customer_id)', 'Hash distribution — high cardinality joins'),
    ('TRUNCATE(2, currency)',   'Prefix pruning — string columns'),
    ('YEAR(transaction_ts)',    'Coarse date pruning — long history'),
    ('DAY(transaction_ts)',     'Fine date pruning — event/log tables'),
], columns=['Transform','When to use'])
display(transforms)

---
## Summary — All 18 Features Complete

**Key takeaways for customers:**

1. **One endpoint, all engines** — Horizon Catalog exposes every Iceberg table at a single REST URI
2. **True open access** — Spark, Flink, Trino, DuckDB, PyIceberg connect with zero proprietary code
3. **Governance that follows the data** — RBAC, row access policies, and masking enforce on every engine
4. **No static AWS keys** — Credential vending provides short-lived STS credentials automatically
5. **Operational excellence** — OPTIMIZE, time travel, schema evolution, Dynamic Tables all work natively
6. **Unique differentiator** — Policy enforcement on external engine queries is unique to Snowflake

**Resources:**
- Companion GitHub repo: `github.com/sfc-gh-venkatmedida/snowflake-iceberg-interop-explorer`
- Interactive Streamlit app: `AFENG_DB.ICEBERG_DEMOS.ICEBERG_INTEROP_EXPLORER` (AFE Americas account)
- Snowflake Iceberg docs: `docs.snowflake.com/en/user-guide/tables-iceberg`

---
## Features 19–25: Secure Sharing · Delta Sharing · Snowpark · Tags · Unity Catalog · PrivateLink · Iceberg v3

In [ ]:
# Feature 19 — Secure Data Sharing for Iceberg (multi-tenant)
print("=== Feature 19: Secure Data Sharing for Iceberg ===")

# Pattern A: Cross-account share
print("Pattern A — Cross-account Iceberg share:")
print("""
CREATE SHARE iceberg_data_share;
GRANT USAGE ON DATABASE horizon_demo_db        TO SHARE iceberg_data_share;
GRANT USAGE ON SCHEMA   horizon_demo_db.public  TO SHARE iceberg_data_share;
GRANT SELECT ON TABLE   horizon_demo_db.public.transactions TO SHARE iceberg_data_share;
ALTER SHARE iceberg_data_share ADD ACCOUNTS = <consumer_account_id>;
""")

# Pattern B: Multi-tenant row isolation
print("Pattern B — Multi-tenant row isolation via RAP:")
session.sql(f"""
CREATE OR REPLACE ROW ACCESS POLICY {DATABASE}.{SCHEMA}.rap_multitenant
AS (tenant_col VARCHAR) RETURNS BOOLEAN ->
    CASE
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN','SYSADMIN') THEN TRUE
        ELSE tenant_col = SYSTEM$CURRENT_ACCOUNT()
    END
""").collect()
print("✓ Multi-tenant RAP created")
print("  Each consumer account sees ONLY its own rows — enforces in Snowflake SQL AND Horizon REST")
print(f"  Current account identity: {session.sql('SELECT SYSTEM$CURRENT_ACCOUNT()').to_pandas().iloc[0,0]}")

# Feature 20 — Delta Sharing
print("\n=== Feature 20: Delta Sharing Protocol ===")
print("Share Iceberg data with non-Snowflake consumers using open Delta Sharing protocol:")
print("""
CREATE SHARE delta_iceberg_share;
GRANT SELECT ON TABLE horizon_demo_db.public.transactions TO SHARE delta_iceberg_share;
CREATE RECIPIENT partner_team COMMENT = 'External analytics team — no Snowflake account needed';
ALTER SHARE delta_iceberg_share ADD RECIPIENTS partner_team;
DESC RECIPIENT partner_team;  -- returns activation link / profile.share file

-- Consumer (Python, no Snowflake):
-- import delta_sharing
-- df = delta_sharing.load_as_pandas("profile.share#delta_iceberg_share.public.transactions")
""")

In [ ]:
# Feature 21 — Snowpark on Iceberg
print("=== Feature 21: Snowpark on Iceberg ===")
from snowflake.snowpark import functions as F

df = session.table(f"{DATABASE}.{SCHEMA}.transactions")
print(f"Snowpark read {df.count()} rows from Iceberg table")

agg = (df.filter(F.col("STATUS") == "COMPLETED")
         .group_by("CURRENCY")
         .agg(F.count("TRANSACTION_ID").alias("TXN_COUNT"),
              F.round(F.sum("AMOUNT"), 2).alias("TOTAL_AMOUNT"))
         .sort(F.col("TOTAL_AMOUNT").desc()))
display(agg.to_pandas())

session.sql(f"""
CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.revenue_summary (
    currency VARCHAR(3), txn_count BIGINT, total_amount DECIMAL(18,2)
) CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}' BASE_LOCATION='horizon_demo/revenue_summary/'
""").collect()
agg.write.mode("overwrite").save_as_table(f"{DATABASE}.{SCHEMA}.revenue_summary")
print("✓ Written to Iceberg via Snowpark — open for external engines via Horizon")

# Feature 22 — Object Tags + Data Classification
print("\n=== Feature 22: Object Tags + Data Classification ===")
session.sql("CREATE TAG IF NOT EXISTS data_sensitivity ALLOWED_VALUES 'PUBLIC','INTERNAL','CONFIDENTIAL','RESTRICTED'").collect()
session.sql("CREATE TAG IF NOT EXISTS data_domain ALLOWED_VALUES 'FINANCIAL','PII','OPERATIONAL','REFERENCE'").collect()
session.sql(f"ALTER TABLE {DATABASE}.{SCHEMA}.transactions SET TAG data_sensitivity='CONFIDENTIAL', data_domain='FINANCIAL'").collect()
session.sql(f"ALTER TABLE {DATABASE}.{SCHEMA}.transactions MODIFY COLUMN customer_id SET TAG data_sensitivity='RESTRICTED', data_domain='PII'").collect()
print("✓ Tags applied to table and customer_id column")
tags = session.sql(f"SELECT * FROM TABLE(information_schema.tag_references('{DATABASE}.{SCHEMA}.transactions','TABLE'))").to_pandas()
display(tags[['TAG_NAME','TAG_VALUE','DOMAIN']].head(5) if len(tags) > 0 else pd.DataFrame({'message':['Tags applied — run SHOW in Snowsight to verify']}))

In [ ]:
# Feature 23 — Unity Catalog ↔ Horizon
print("=== Feature 23: Unity Catalog ↔ Horizon Catalog ===")
print("""
Direction A — Databricks reads Snowflake Iceberg via Horizon:
  spark.sql.catalog.snowflake.type      = rest
  spark.sql.catalog.snowflake.uri       = {HORIZON_URI}
  spark.sql.catalog.snowflake.token     = <service_principal_token>
  spark.sql.catalog.snowflake.warehouse = horizon_demo_db
  → spark.table("snowflake.public.transactions").show()

Direction B — Snowflake reads Databricks Unity Catalog:
  CREATE CATALOG INTEGRATION unity_catalog_int
      CATALOG_SOURCE = ICEBERG_REST
      REST_CONFIG = ( CATALOG_URI = 'https://<workspace>.azuredatabricks.net/api/2.1/unity-catalog/iceberg' ...)
  CREATE DATABASE uc_db LINKED_CATALOG = (CATALOG_INTEGRATION = 'unity_catalog_int') ...
  SELECT * FROM uc_db.main.my_delta_table LIMIT 10;
""".format(HORIZON_URI=HORIZON_URI))

session.sql("SHOW INTEGRATIONS LIKE '%UNITY%'").to_pandas()  # check if already configured

# Feature 24 — PrivateLink
print("\n=== Feature 24: PrivateLink for Catalog Integrations ===")
print("""
Inbound PrivateLink (external engines → Horizon):
  Use: https://<account>.privatelink.snowflakecomputing.com/polaris/api/catalog
  (instead of: https://<account>.snowflakecomputing.com/polaris/api/catalog)

Outbound PrivateLink (Snowflake → Glue/Polaris/UC):
  CATALOG_URI = 'https://glue.us-east-2.vpce.amazonaws.com/iceberg'
  (private endpoint DNS, not the public URL)

Network Policy to restrict Horizon to private IPs only:
  CREATE NETWORK POLICY horizon_private ALLOWED_IP_LIST = ('10.0.0.0/8', '172.16.0.0/12');
  ALTER ACCOUNT SET NETWORK_POLICY = horizon_private;

Required for: HIPAA, PCI-DSS, FedRAMP, financial services customers
""")

In [ ]:
# Feature 25 — Iceberg v3 Features
print("=== Feature 25: Iceberg v3 Features ===")
session.sql(f"""
CREATE OR REPLACE ICEBERG TABLE {DATABASE}.{SCHEMA}.events_v3 (
    event_id        VARCHAR(36),
    user_id         VARCHAR(36),
    event_type      VARCHAR(50),
    event_ts        TIMESTAMP_NTZ(9),          -- nanosecond precision (v3)
    payload         OBJECT,                     -- semi-structured JSON (v3)
    tags            ARRAY,                      -- array type (v3)
    severity        INT     DEFAULT 0,          -- default value (v3)
    is_processed    BOOLEAN DEFAULT FALSE,      -- default value (v3)
    region          VARCHAR(50)
)   CATALOG='SNOWFLAKE' EXTERNAL_VOLUME='{EXTERNAL_VOLUME}' BASE_LOCATION='horizon_demo/events_v3/'
""").collect()

session.sql(f"""
INSERT INTO {DATABASE}.{SCHEMA}.events_v3 (event_id, user_id, event_type, event_ts, payload, tags, severity, region)
VALUES
    ('evt-001','usr-A','purchase','2024-01-15 09:30:00.123456789'::TIMESTAMP_NTZ(9),
     OBJECT_CONSTRUCT('product_id','P123','price',99.99),ARRAY_CONSTRUCT('high-value','first-purchase'),2,'us-west'),
    ('evt-002','usr-B','click',   '2024-01-15 09:30:00.987654321'::TIMESTAMP_NTZ(9),
     OBJECT_CONSTRUCT('page','/home','duration_ms',3200),ARRAY_CONSTRUCT('mobile'),0,'eu-west')
""").collect()

df_v3 = session.sql(f"""
SELECT event_id, event_ts, payload:product_id::VARCHAR AS product_id,
       payload:price::DECIMAL(10,2) AS price, tags[0]::VARCHAR AS first_tag,
       severity, is_processed
FROM {DATABASE}.{SCHEMA}.events_v3
""").to_pandas()
display(df_v3)

v3_summary = pd.DataFrame([
    ('TIMESTAMP_NTZ(9)',  'Nanosecond precision',       'Event/telemetry, IoT sensors'),
    ('OBJECT/ARRAY',      'Semi-structured JSON columns','Log data, API payloads'),
    ('DEFAULT <expr>',    'Default column values',       'Audit fields, flags, counters'),
    ('Position deletes',  'Efficient row-level deletes', 'High-frequency UPDATE/DELETE'),
    ('Row lineage',       'METADATA$ columns',           'Audit trail, data provenance'),
], columns=['v3 Feature','What it enables','Use case'])
display(v3_summary)
print("Engines supporting v3: Spark ≥3.4+Iceberg 1.5, PyIceberg ≥0.7, Trino ≥438, DuckDB ≥1.2 (read)")

---
## Complete Snowflake Iceberg Interoperability Story — 25 Features

| Category | Features |
|----------|---------|
| **Core interop** | 1 Open REST, 2 Polaris, 3 Single Endpoint |
| **Engine access** | 4 Read+Write, 9 Engines (8 supported) |
| **Governance** | 5 Security Model, 6 Credential Vending, 7 Multi-Engine, 8 Policies, 22 Tags |
| **Storage** | 10 Snowflake Storage, 11 Glue+CLD |
| **Data management** | 12 Time Travel, 13 Maintenance, 14 Schema Evolution |
| **Pipelines** | 16 Streaming, 17 Dynamic Tables |
| **Performance** | 18 Partitioning |
| **Sharing** | 19 Secure Data Sharing, 20 Delta Sharing |
| **Native Python** | 21 Snowpark on Iceberg |
| **Multi-cloud** | 23 Unity Catalog ↔ Horizon, 24 PrivateLink |
| **Format** | 25 Iceberg v3 |
| **Reference** | 15 Competitive Positioning |

**This notebook covers the complete Snowflake Iceberg interop story.**
Clone + share: `github.com/sfc-gh-venkatmedida/snowflake-iceberg-interop-explorer`